# African NLP Models: What Exists and Who Built It

**Computational Text Analysis Pack · Module 23 — African NLP Models and Benchmarks**

*Technique:* The AfriBERTa/AfroXLMR/LaBSE landscape; pretrained transfer vs training from scratch

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/13-african-nlp-models/23a_african_nlp_models.ipynb)

Data: `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

Every method in the first 22 modules learned everything it knew from the corpus
you handed it. A pretrained model arrives **already knowing the language**, from
text you never supplied, and that single difference is what the whole neural
turn amounts to for a humanities researcher.

This notebook is the map: which models exist for African languages, who built
them, what they were trained on, and — the part that decides whether they are
usable for your project — **which languages they actually cover.**

## The tooling

**HuggingFace `transformers`** is the working layer for every model here: it
downloads the weights, tokenises, and runs the forward pass. Underneath it sits
**PyTorch** (the research default, dynamic and easy to debug); **TensorFlow /
Keras** is the other option, and what the Case Studies module uses.

The models themselves come from African NLP research groups:

- **AfriBERTa** (Ogueji et al.) — trained on 11 African languages, deliberately
  small, and evidence that useful models do not need web-scale data.
- **AfroXLMR** (Alabi et al.) — XLM-R adapted to African languages; the NER model
  in this notebook is its fine-tuned descendant.
- **LaBSE** (Google) — 109 languages, sentence-level, and the reason a Hausa
  sentence and its English translation land in the same place.
- **MasakhaNER / MasakhaNEWS / MasakhaPOS** — the benchmarks, built by the
  Masakhane community. `masakhaner.csv` in this course *is* that benchmark.

### Where the GPU comes from, and why you never need a key

The setup cell picks a backend from wherever you are running:

| Running in | GPU | What you need |
|---|---|---|
| **CourseHub Lab** | a shared L40S, reached through an authenticated CourseHub endpoint | nothing — the credential stays on the server |
| **Colab** | *your own* free GPU (`Runtime > Change runtime type > T4`) | nothing |
| **Anywhere offline** | none; cells use vectors shipped with the course | nothing |

There is no API key in this course, by design. A notebook cell is readable by
whoever runs it, so a key pasted into one is a key given away.

In [ ]:
import sys, os

IN_COLAB = "google.colab" in sys.modules
BACKEND = "cached"
_pod = _hf = None

if IN_COLAB:
    # Colab hands every participant their own GPU. Use it rather than sharing
    # one server: it is faster for a room full of people, and needs no key.
    try:
        import transformers  # noqa: F401
    except ImportError:
        os.system("pip install -q transformers")
    import torch
    BACKEND = "colab-gpu" if torch.cuda.is_available() else "colab-cpu"
else:
    try:
        sys.path.insert(0, "/app")      # the CourseHub runner executes inside /app
        from utils import gpu_pod as _pod
        _info = _pod.health()
        BACKEND = "coursehub-pod"
    except Exception as exc:
        _why = str(exc)[:80]

POD_LIVE = BACKEND in ("coursehub-pod", "colab-gpu", "colab-cpu")

def _hf_pipe(kind, repo):
    global _hf
    _hf = _hf or {}
    if (kind, repo) not in _hf:
        from transformers import AutoModel, AutoTokenizer, pipeline
        import torch
        dev = 0 if torch.cuda.is_available() else -1
        if kind == "ner":
            _hf[(kind, repo)] = pipeline("token-classification", model=repo,
                                         aggregation_strategy="simple", device=dev)
        else:
            _hf[(kind, repo)] = (AutoTokenizer.from_pretrained(repo),
                                 AutoModel.from_pretrained(repo))
    return _hf[(kind, repo)]

def embed(texts, model="labse"):
    """Sentence embeddings, wherever we are. Returns a numpy array."""
    import numpy as np
    if BACKEND == "coursehub-pod":
        return np.asarray(_pod.embed(list(texts), model=model)["vectors"], dtype="float32")
    if BACKEND.startswith("colab"):
        import torch
        tok, mod = _hf_pipe("embed", "sentence-transformers/LaBSE")
        if torch.cuda.is_available():
            mod = mod.cuda()
        with torch.no_grad():
            b = tok(list(texts), padding=True, truncation=True,
                    max_length=128, return_tensors="pt")
            b = {k: v.to(mod.device) for k, v in b.items()}
            out = mod(**b).last_hidden_state
            m = b["attention_mask"].unsqueeze(-1).float()
            v = (out * m).sum(1) / m.sum(1).clamp(min=1e-9)
            v = torch.nn.functional.normalize(v, dim=-1)
        return v.cpu().numpy().astype("float32")
    raise RuntimeError("no GPU backend available; use the shipped vectors instead")

def ner(texts, model="masakhaner"):
    """Named entities from the MasakhaNER 2.0 benchmark model."""
    repo = "masakhane/afroxlmr-large-ner-masakhaner-1.0_2.0"
    if BACKEND == "coursehub-pod":
        return _pod.ner(list(texts), model=model)["results"]
    if BACKEND.startswith("colab"):
        pipe = _hf_pipe("ner", repo)
        out = pipe(list(texts))
        if out and isinstance(out[0], dict):
            out = [out]
        return [[{"text": e["word"], "label": e["entity_group"],
                  "score": round(float(e["score"]), 4)} for e in ents] for ents in out]
    raise RuntimeError("no GPU backend available")

print(f"backend: {BACKEND}")
if BACKEND == "coursehub-pod":
    print(f"   {_info['gpu']}, {_info['vram_total_mb']:,} MB VRAM (shared, via CourseHub)")
elif BACKEND == "colab-gpu":
    import torch; print(f"   {torch.cuda.get_device_name(0)} (your own Colab GPU)")
elif BACKEND == "colab-cpu":
    print("   Colab without a GPU — Runtime > Change runtime type > T4 GPU makes this faster")
else:
    print(f"   no live backend ({_why}); cells fall back to the vectors shipped with the course")

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. It does not
    match combining marks, and a tone mark on ọ or ẹ is always one, so the
    pattern adds the combining-accent range U+0300-U+036F; without it ọ̀rọ̀
    splits into ọ + rọ. The common shortcut `[a-z]+` throws away far more and
    quietly changes every count in this notebook — see notebook 03b.
    """
    return re.findall(r"[\w\u0300-\u036f]+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé lórí ọ̀rọ̀ ẹ̀kọ́ — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("bare \\w+   :", re.findall(r"\w+", sample.lower()))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter; bare \\w+")
print("splits them wherever a tone mark sits on ọ or ẹ.")

## Why pretraining matters, in one measurement

Module 7 trained a classifier on MasakhaNEWS from scratch and reached about
0.81. The Case Studies module got 0.771 from a small neural model. Both learned
the language *only* from the 8,400 documents in this corpus.

Below is what a corpus of this size looks like next to what a pretrained model
consumed. The gap is the argument.

In [ ]:
corpus_words = int(df["n_words"].sum())
scales = [
    ("This course's news corpus", corpus_words),
    ("All three ADH corpora", 7_000_000),
    ("AfriBERTa training data", 1_000_000_000),
    ("XLM-R / AfroXLMR base data", 295_000_000_000),
]
print(f"{'corpus':<32}{'words':>18}{'x ours':>12}")
print("-" * 62)
for name, n in scales:
    print(f"{name:<32}{n:>18,}{n / corpus_words:>11,.0f}x")
print()
print("You are not going to out-train that. The neural workflow for DH is to")
print("start from someone else's pretraining and adapt — never from scratch.")

## Which languages are actually covered

This is the question that decides whether any of this is usable for your
research, and the answer is uncomfortable. Coverage follows digitised text,
which follows colonial language policy and commercial value.

In [ ]:
# Languages in this course's corpora, against what the main models cover.
AFRIBERTA = {"Amharic", "Hausa", "Igbo", "Nigerian Pidgin", "Somali", "Swahili",
             "Tigrinya", "Yoruba", "Oromo", "Kinyarwanda"}
AFROXLMR = AFRIBERTA | {"Shona", "Xhosa", "Zulu", "Chichewa", "Luganda",
                        "Setswana", "Wolof", "French", "English"}
LABSE_ISH = AFROXLMR | {"Lingala", "Rundi"}

langs = sorted(df["language"].unique())
rows = []
for lang in langs:
    rows.append({"language": lang,
                 "docs": int((df["language"] == lang).sum()),
                 "AfriBERTa": "yes" if lang in AFRIBERTA else "-",
                 "AfroXLMR": "yes" if lang in AFROXLMR else "-",
                 "LaBSE": "yes" if lang in LABSE_ISH else "-"})
cover = pd.DataFrame(rows)
print(cover.to_string(index=False))
print()
n_all = sum(1 for r in rows if r["AfriBERTa"] == "yes")
print(f"{n_all} of {len(rows)} languages in this corpus are covered by AfriBERTa.")
print("Africa has over 2,000 languages. Every model above covers under 1% of them.")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
counts = [sum(1 for r in rows if r[m] == "yes") for m in ["AfriBERTa", "AfroXLMR", "LaBSE"]]
ax.bar(["AfriBERTa", "AfroXLMR", "LaBSE"], counts, color="steelblue")
ax.axhline(len(rows), ls="--", color="grey", lw=1)
ax.annotate("languages in this corpus", xy=(0, len(rows)), fontsize=8,
            xytext=(0, len(rows) + 0.3), color="grey")
ax.set_ylabel("languages covered")
ax.set_title("Model coverage against one small corpus")
plt.tight_layout()

### 🔧 Your turn 1

Find the language in the table with the most documents but the weakest model coverage. What would a project on that language have to do differently?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

It would have to either fine-tune from a multilingual model that has *some*
related-language signal, or fall back on the classical methods from modules 2-19
— which is exactly why this course teaches those first and why they are not a
historical curiosity. For a language outside every pretrained model, a
well-built frequency list, a keyness comparison and a hand-written finite-state
morphology from module 14 will get further than a transformer will, and they are
auditable in a way it is not.

</details>